[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/01_state_and_motion_of_water.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 01. The state and motion of water in a geothermal reservoir

Lecture 1, Monday 5 October 2026. This notebook has an **in-class part** (Parts 1 to 4) and a **take-home part** (Assignment 1, at the end).

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** Replace every `___` with code. A test cell checks your answer.
- **C. Extensions.** Open questions for when you finish early. Several feed into Assignment 1.

**Rules for this course**

1. Write a **prediction** before you run a cell that computes something new.
2. Keep **units** next to every number in your code comments.
3. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. Setup, units and water properties
2. Enthalpy, flow work and two-phase mixtures
3. Pressure with depth and the steam fraction of rising fluid
4. Why fluid flows. Vertical and horizontal pressure gradients
5. Take-home Assignment 1

## Part 1. Setup, units and water properties

We use the `iapws` package, which implements IAPWS-IF97, the international standard for water and steam properties.

Units in `iapws` are pressure in **MPa**, temperature in **K**, specific enthalpy and internal energy in **kJ/kg**, density in kg/m³ and viscosity in Pa s. In this notebook we give temperature in °C and convert inside the helper functions.

In [ ]:
# Install the water property package.  Runs once per Colab session, takes about 20 seconds.
%pip install -q iapws

# If a later notebook needs data files from the course repository, fetch them like this:
#   !wget -q https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/<file>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from iapws import IAPWS97, IAPWS95

plt.rcParams.update({"font.size": 13, "figure.figsize": (8, 5), "axes.grid": True})

G = 9.81                   # gravitational acceleration, m/s2
P_ATM = 0.101325           # atmospheric pressure, MPa
K0 = 273.15                # 0 C expressed in K
P_CRIT = IAPWS95.Pc        # critical pressure, MPa
T_CRIT = IAPWS95.Tc - K0   # critical temperature, C
print(f"Critical point: {P_CRIT} MPa, {T_CRIT:.3f} C")

In [ ]:
def water(P, T):
    '''Single-phase water at pressure P (MPa) and temperature T (C).
    Returns rho (kg/m3), u and h (kJ/kg), mu (Pa s), cp (kJ/kg/K) and the phase name.'''
    w = IAPWS97(P=P, T=T + K0)
    return {"rho": w.rho, "u": w.u, "h": w.h, "mu": w.mu, "cp": w.cp, "phase": w.phase}


def saturation(P):
    '''Boiling-curve properties at pressure P (MPa), valid below the critical pressure.
    Returns T (C) and saturated liquid (l) and vapour (v) density, enthalpy and viscosity.'''
    liq = IAPWS97(P=P, x=0)
    vap = IAPWS97(P=P, x=1)
    return {"T": liq.T - K0, "rho_l": liq.rho, "rho_v": vap.rho,
            "h_l": liq.h, "h_v": vap.h, "mu_l": liq.mu, "mu_v": vap.mu}


print(water(10, 300))
print(saturation(1.0))

### B1. Units and a first look at enthalpy

1. Pressure at 1 km depth in a well full of cold water, $P = P_0 + \rho g z$, where $P_0$ is atmospheric pressure (Pa), $\rho$ is density (kg/m³), $g$ is gravitational acceleration (m/s²) and $z$ is depth (m).
2. Energy to heat 1 kg of water from 10 °C to 300 °C at 10 MPa. First with a constant heat capacity, $\Delta h = c_p \Delta T$, where $c_p = 4.2$ kJ/kg/K and $\Delta T$ is the temperature change (K). Then with the steam tables.

**Prediction.** The constant-$c_p$ estimate is ______ (too high / too low) by about ______ %.

In [ ]:
# 1. Pressure at the bottom of a 1000 m cold water column
rho_cold = 1000.0          # kg/m3
depth = 1000.0             # m
P_bottom_MPa = ___         # MPa

# 2. Heating 1 kg from 10 C to 300 C at 10 MPa
dh_cp = ___                # kJ/kg, constant cp = 4.2 kJ/kg/K
dh_table = ___             # kJ/kg, from water()

print(f"P = {P_bottom_MPa:.2f} MPa = {10 * P_bottom_MPa:.1f} bar")
print(f"constant cp: {dh_cp:.0f} kJ/kg, steam tables: {dh_table:.0f} kJ/kg")

In [ ]:
# Test
assert abs(P_bottom_MPa - 9.91) < 0.02, "1 MPa = 1e6 Pa"
assert abs(dh_table - 1291) < 3
print("B1 OK")

## Part 2. Enthalpy, heat capacity and latent heat

Heat one kilogram of water at constant pressure. The first law is $dU = \delta Q - P\,dV$, and at constant pressure $P\,dV = d(PV)$, so

$$(\delta q)_P = dh, \qquad h = u + \frac{P}{\rho}$$

where $(\delta q)_P$ is the heat added per kilogram at constant pressure (J/kg), $h$ is specific enthalpy (J/kg), $u$ is specific internal energy (J/kg), $P$ is pressure (Pa) and $\rho$ is density (kg/m³). Enthalpy is the heat you must add at constant pressure. The term $P/\rho$ is the volume work carried with the fluid.

**Prediction.** At 1 MPa, heating liquid water from 10 °C to its boiling point takes ______ kJ/kg, and boiling it there takes ______ kJ/kg.

### B2.1 The energy budget of heating and boiling

In [ ]:
P = 1.0                                   # MPa
cold = water(P, 10)                       # liquid at 10 C
sat = saturation(P)                       # boiling curve at 1 MPa
print(f"boiling temperature at {P} MPa: {sat['T']:.1f} C")

sensible = ___                            # kJ/kg, heating liquid from 10 C to the boiling point
latent = ___                              # kJ/kg, boiling at constant pressure

# How much of each is volume work P dv?  v = 1/rho, so P dv is in kJ/kg when P is in MPa and rho in kg/m3
pdv_liquid = ___                          # kJ/kg
pdv_boiling = ___                         # kJ/kg

print(f"sensible heat {sensible:.0f} kJ/kg (P dv = {pdv_liquid:.2f} kJ/kg)")
print(f"latent heat   {latent:.0f} kJ/kg (P dv = {pdv_boiling:.0f} kJ/kg)")

In [ ]:
# Test
assert abs(sensible - 720) < 3 and abs(latent - 2014) < 3
assert abs(pdv_boiling - 193) < 3, "P dv: keep MPa and kg/m3 consistent"
print("B2.1 OK")

Boiling costs about three times as much energy as heating the liquid all the way from 10 °C, and about 10 % of the latent heat is spent pushing the surroundings aside as the steam expands. That term is why we book-keep with $h$ and not with $u$.

### B2.2 Heat capacity is the slope of enthalpy

$$c_p = \left(\frac{\partial h}{\partial T}\right)_P, \qquad \Delta h = \int_{T_1}^{T_2} c_p\, dT$$

where $c_p$ is the isobaric specific heat capacity (J/kg/K) and $T$ is temperature (K). Using $\Delta h = c_p \Delta T$ with one value of $c_p$ assumes $c_p$ is constant.

**Prediction.** Heating 1 kg of water from 10 to 300 °C at 10 MPa with a constant $c_p$ of 4.2 kJ/kg/K is wrong by about ______ %.

In [ ]:
T_grid_liq = np.linspace(10, 300, 200)                 # C
cp_values = np.array([___ for T in T_grid_liq])        # kJ/kg/K at 10 MPa

dh_constant_cp = 4.2 * (300 - 10)                      # kJ/kg
dh_integral = np.trapezoid(cp_values, T_grid_liq)      # kJ/kg (np.trapz on older numpy)
dh_table = ___                                         # kJ/kg, straight from enthalpy

print(f"constant cp: {dh_constant_cp:.0f} kJ/kg")
print(f"integral of cp: {dh_integral:.0f} kJ/kg")
print(f"from the tables: {dh_table:.0f} kJ/kg")

plt.plot(T_grid_liq, cp_values)
plt.axhline(4.2, ls="--", color="gray")
plt.xlabel("temperature (C)"); plt.ylabel("cp of liquid at 10 MPa (kJ/kg/K)")
plt.show()

In [ ]:
# Test
assert abs(dh_integral - dh_table) < 15, "The integral of cp should reproduce the enthalpy difference"
print(f"constant cp is off by {100 * (dh_constant_cp - dh_table) / dh_table:+.0f} %")
print("B2.2 OK")

### A. The pressure-enthalpy diagram

Inside the two-phase dome, liquid and steam coexist at the same pressure and temperature. The steam mass fraction of a mixture with enthalpy $h$ at pressure $P$ is

$$x = \frac{h - h_l(P)}{h_v(P) - h_l(P)}$$

where $x$ is the steam mass fraction (dimensionless), and $h_l(P)$ and $h_v(P)$ are the specific enthalpies of saturated liquid and saturated vapour at pressure $P$ (kJ/kg). The dashed lines below are lines of constant $x$.

In [ ]:
def ph_diagram(ax=None, isotherms=(100, 200, 300, 350, 400, 500)):
    '''Draw the two-phase dome, lines of constant steam fraction and isotherms on a P-h diagram.'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    P_dome = np.linspace(0.01, P_CRIT * 0.9999, 300)
    sat = [saturation(p) for p in P_dome]
    hl = np.array([s["h_l"] for s in sat]); hv = np.array([s["h_v"] for s in sat])
    ax.plot(hl, P_dome, "k", lw=2); ax.plot(hv, P_dome, "k", lw=2)
    for xq in [0.2, 0.4, 0.6, 0.8]:
        ax.plot(hl + xq * (hv - hl), P_dome, "g--", lw=0.8)
    for T in isotherms:
        P_line = np.linspace(0.02, 40, 400)
        h_line = []
        for p in P_line:
            if p < P_CRIT and abs(saturation(p)["T"] - T) < 1.0:
                h_line.append(np.nan)            # skip the flat two-phase part
            else:
                try:
                    h_line.append(water(p, T)["h"])
                except Exception:
                    h_line.append(np.nan)
        ax.plot(h_line, P_line, lw=1, label=f"{T} C")
    ax.set_xlabel("Specific enthalpy h (kJ/kg)"); ax.set_ylabel("Pressure P (MPa)")
    ax.set_xlim(0, 3600); ax.set_ylim(0, 40)
    return ax


ax = ph_diagram()
ax.legend(title="isotherms", fontsize=10)
plt.show()

### B2.3 Steam mass fraction and steam volume fraction

The steam volume fraction (saturation) is

$$S = \frac{x/\rho_v}{x/\rho_v + (1-x)/\rho_l}$$

where $S$ is the fraction of the volume filled by steam (dimensionless), and $\rho_l$ and $\rho_v$ are the densities of saturated liquid and saturated vapour at the same pressure (kg/m³).

**Prediction.** At 1 MPa, a mixture with 1 % steam by mass has $S \approx$ ______.

In [ ]:
def steam_fraction(h, P):
    '''Steam mass fraction x (-) for enthalpy h (kJ/kg) at pressure P (MPa).
    Values below 0 mean compressed liquid, above 1 mean superheated steam.'''
    s = saturation(P)
    return ___


def volume_fraction(x, P):
    '''Steam volume fraction S (-) for steam mass fraction x (-) at pressure P (MPa).'''
    s = saturation(P)
    return ___


print(f"x for h = 1400 kJ/kg at 1 MPa: {steam_fraction(1400, 1.0):.3f}")
print(f"S for x = 0.01 at 1 MPa: {volume_fraction(0.01, 1.0):.3f}")

In [ ]:
# Test
assert abs(steam_fraction(1400, 1.0) - 0.316) < 0.003
assert abs(volume_fraction(0.01, 1.0) - 0.635) < 0.005
print("B2.3 OK")

### C2. Extensions

1. Plot $S$ against $x$ at 0.5, 5 and 20 MPa. Explain the shape in one sentence using the densities.
2. Compare $u$, $P/\rho$ and $h$ for liquid at 20 °C and 0.1 MPa, liquid at 250 °C and 10 MPa, and steam at 400 °C and 1 MPa. For which state is the flow work a large part of $h$?
3. Liquid water at 300 °C and 10 MPa. Compare density and viscosity with water at 20 °C and 0.1 MPa. Which ratio matters for buoyancy, and which for how easily fluid moves through rock?
4. Plot $c_p$ from 300 to 600 °C at 23, 25 and 30 MPa. What happens near the critical point, and why does it matter for superhot wells?

---
**Break.** Parts 3 and 4 follow the board session on pressure, ascent and flow.

## Part 3. Pressure with depth and the steam fraction of rising fluid

### Hydrostatic pressure and the boiling point with depth (BPD)

$$\frac{dP}{dz} = \rho(P,T)\,g, \qquad P_{i+1} = P_i + \rho(P_i, T_i)\,g\,\Delta z$$

where $z$ is depth below the water table (m, positive downward), $\rho(P,T)$ is density at the local pressure and temperature (kg/m³), $i$ is the index of a depth level and $\Delta z$ is the depth step (m).

For the BPD curve, water is at its boiling point at every depth:

$$T_i = T_{sat}(P_i), \qquad \rho_i = \rho_l^{sat}(P_i)$$

where $T_{sat}(P)$ is the boiling temperature at pressure $P$ and $\rho_l^{sat}(P)$ is the density of saturated liquid at $P$.

**Prediction.** BPD temperature at 1000 m depth: ______ °C.

### Clausius-Clapeyron: how boiling temperature follows pressure

Along the boiling curve liquid and vapour have the same specific Gibbs energy. Taking a small step along the curve gives

$$\frac{dP_{sat}}{dT} = \frac{s_v - s_l}{v_v - v_l} = \frac{L}{T\,(v_v - v_l)}$$

where $P_{sat}$ is the boiling pressure (Pa), $T$ is the boiling temperature (K), $s_l$ and $s_v$ are the specific entropies of saturated liquid and vapour (J/kg/K), $v = 1/\rho$ is specific volume (m³/kg) and $L$ is the latent heat of vaporisation (J/kg).

### B3.0 Check Clausius-Clapeyron against the tables

**Prediction.** At 250 °C the boiling curve rises by about ______ kPa per K.

In [ ]:
def clapeyron_slope(T_C):
    '''dP_sat/dT (Pa/K) at boiling temperature T_C (C), from L and the volume change.'''
    liq = IAPWS97(T=T_C + K0, x=0)
    vap = IAPWS97(T=T_C + K0, x=1)
    L = ___                                   # J/kg
    dv = ___                                  # m3/kg
    return ___


def numerical_slope(T_C, dT=0.5):
    '''dP_sat/dT (Pa/K) by differentiating the IF97 saturation curve.'''
    return (IAPWS97(T=T_C + dT + K0, x=0).P - IAPWS97(T=T_C - dT + K0, x=0).P) * 1e6 / (2 * dT)


for T_C in [150, 250, 350]:
    print(f"{T_C} C: Clapeyron {clapeyron_slope(T_C)/1e3:7.2f} kPa/K, numerical {numerical_slope(T_C)/1e3:7.2f} kPa/K")

In [ ]:
# Test
assert abs(clapeyron_slope(250) - numerical_slope(250)) / numerical_slope(250) < 0.01
print("B3.0 OK")

### B3.1 The BPD curve by stepping down

Watch the units. $\rho g \Delta z$ is in Pa, and our pressure is in MPa. Stop when the pressure reaches the critical pressure.

In [ ]:
def bpd_curve(dz=1.0, z_max=4000.0, P_top=P_ATM, z_top=0.0):
    '''Boiling point with depth for pure water.
    dz: depth step (m).  z_max: maximum depth (m).
    P_top: pressure (MPa) at depth z_top (m), for example the water table.
    Returns arrays z (m), P (MPa), T (C).'''
    z_list, P_list = [z_top], [P_top]
    while z_list[-1] < z_max:
        rho_l = ___                                  # kg/m3, saturated liquid at the current pressure
        P_next = ___                                 # MPa
        if P_next >= P_CRIT:
            break
        z_list.append(z_list[-1] + dz)
        P_list.append(P_next)
    P_arr = np.array(P_list)
    T_arr = np.array([saturation(p)["T"] for p in P_arr])
    return np.array(z_list), P_arr, T_arr


z_b, P_b, T_b = bpd_curve(dz=5.0)
for d in [500, 1000, 2000]:
    print(f"{d:5d} m: P = {np.interp(d, z_b, P_b):6.2f} MPa, T = {np.interp(d, z_b, T_b):5.1f} C")

In [ ]:
# Test
assert abs(np.interp(1000, z_b, P_b) - 7.98) < 0.05
assert abs(np.interp(1000, z_b, T_b) - 294.9) < 1.0
print("B3.1 OK")

### B3.2 Steam fraction of fluid rising along the BPD curve

For steady flow with no shaft work, the energy equation along a rising path is

$$h(z) = h_1 - \frac{v^2 - v_1^2}{2} - g\,(z_1 - z) - \frac{\dot{Q}_{loss}}{\dot{m}}$$

where $h_1$ and $v_1$ are the specific enthalpy (J/kg) and velocity (m/s) at the starting depth $z_1$ (m), $h(z)$ and $v$ are the values at shallower depth $z$, $g\,(z_1-z)$ is the potential energy gained per kilogram (J/kg), $\dot{Q}_{loss}$ is the heat lost to the surroundings between $z_1$ and $z$ (W) and $\dot{m}$ is the mass flow rate (kg/s).

Here we keep two versions:

- **isenthalpic**, $h(z) = h_1$;
- **with height**, $h(z) = h_1 - g\,(z_1 - z)$.

Saturated liquid at 300 °C rises from the depth where the BPD pressure equals $P_{sat}(300\ °C)$. Remember that $h$ is in kJ/kg in `iapws`, so divide $g\,\Delta z$ by 1000.

**Prediction.** Steam mass fraction at the surface: ______.

In [ ]:
T_start = 300.0                                          # C
h1 = IAPWS97(T=T_start + K0, x=0).h                      # kJ/kg, saturated liquid
P_onset = IAPWS97(T=T_start + K0, x=0).P                 # MPa, boiling starts here
z1 = np.interp(P_onset, P_b, z_b)                        # m, depth where BPD pressure = P_onset
print(f"h1 = {h1:.1f} kJ/kg, boiling starts at {z1:.0f} m")

z_up = np.linspace(z1, 0, 200)                           # m, from z1 up to the surface
P_up = np.interp(z_up, z_b, P_b)                         # MPa, pressure follows the BPD curve

x_iso = np.array([___ for p in P_up])                                    # isenthalpic
x_gz = np.array([___ for zz, p in zip(z_up, P_up)])                      # with height
S_iso = np.array([volume_fraction(max(x, 0), p) for x, p in zip(x_iso, P_up)])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
a1.plot(x_iso, z_up, label="h constant"); a1.plot(x_gz, z_up, "--", label="h + gz constant")
a1.set_xlabel("steam mass fraction x (-)"); a1.set_ylabel("depth (m)"); a1.invert_yaxis(); a1.legend()
a2.plot(S_iso, z_up); a2.set_xlabel("steam volume fraction S (-)")
plt.show()
print(f"surface: x = {x_iso[-1]:.3f} (h constant), {x_gz[-1]:.3f} (with height)")

In [ ]:
# Test
assert abs(z1 - 1087) < 10
assert abs(x_iso[-1] - 0.410) < 0.005 and abs(x_gz[-1] - 0.405) < 0.005
print("B3.2 OK")

### A. The same path on the P-h diagram

In [ ]:
ax = ph_diagram()
ax.plot(np.full_like(P_up, h1), P_up, "r-", lw=3, label="h constant")
ax.plot(h1 - G * (z1 - z_up) / 1e3, P_up, "b--", lw=2, label="h + gz constant")
ax.set_xlim(500, 2000); ax.set_ylim(0, 12); ax.legend()
plt.show()

### C3.0 Optional: the ideal-gas form of Clausius-Clapeyron

If the steam is treated as an ideal gas and $v_l$ is neglected, $v_v = RT/(MP)$, so

$$\frac{d\ln P_{sat}}{dT} = \frac{L M}{R T^2} \qquad\Longrightarrow\qquad \ln P_{sat} = \ln P_{ref} - \frac{L M}{R}\left(\frac{1}{T} - \frac{1}{T_{ref}}\right)$$

where $R = 8.3145$ J/mol/K is the gas constant, $M = 0.018015$ kg/mol is the molar mass of water, and $P_{ref}$ is the boiling pressure at a reference temperature $T_{ref}$ (K). Plot $\ln P_{sat}$ against $1/T$ for IF97 and for this straight line, using $L$ at 175 °C and $T_{ref}$ = 373.15 K. Where does it fail, and why? (Assignment 1, Problem 3.)

### C3. Extensions

1. **Heat loss.** Add a heat loss per unit length $q_\ell$ (W/m), so that $h(z) = h_1 - g\,(z_1-z)/1000 - q_\ell\,(z_1 - z)/(1000\,\dot{m})$ with $h$ in kJ/kg and $\dot{m}$ in kg/s. Try $q_\ell = 150$ W/m with $\dot{m} = 5$ and 50 kg/s.
2. **The separator.** Use the same `steam_fraction` function at a separator pressure of 1.0 MPa. Why is this the same calculation as the ascent?
3. **Step size.** Run `bpd_curve` with `dz` = 200, 100, 50, 10 and 1 m. How does the pressure at 1000 m change? (Assignment 1, Problem 3.)
4. **Slope of the BPD curve.** Differentiate your BPD curve to get $dT/dz$ in °C/km and plot it against depth. Compare with $dT/dz = \rho_l g T (v_v - v_l)/L$. Why is the curve steep near the surface and flat at depth?

## Part 4. Why fluid flows. Vertical and horizontal pressure gradients

Darcy's law in pressure form is

$$\mathbf{q} = -\frac{k}{\mu}\left(\nabla P - \rho\,\mathbf{g}\right)$$

where $\mathbf{q}$ is the Darcy flux (m/s), $k$ is permeability (m²), $\mu$ is dynamic viscosity (Pa s), $\nabla P$ is the pressure gradient (Pa/m), $\rho$ is fluid density (kg/m³) and $\mathbf{g}$ is the gravity vector (m/s², pointing down). Its vertical and horizontal components are

$$q_{up} = \frac{k}{\mu}\left(\frac{\partial P}{\partial z} - \rho g\right), \qquad q_x = -\frac{k}{\mu}\frac{\partial P}{\partial x}$$

where $q_{up}$ is the upward flux (m/s), $z$ is depth (m, positive down), $q_x$ is the horizontal flux (m/s) and $x$ is horizontal distance at constant elevation (m).

A hydrostatic column has $\partial P/\partial z = \rho g$, so $q_{up} = 0$. Flow needs a departure from hydrostatic.

### B4.1 Two columns: the buoyancy-driven pressure difference

Compare a cold column (10 °C) with a hot column whose temperature is $T = \min(T_{sat}(P), T_{res})$, with $T_{res} = 250$ °C. Both have the water table at the surface. The pressure difference at the base is

$$\Delta P_b = \int_0^H (\rho_{c} - \rho_{h})\,g\,dz$$

where $H$ is the column height (m), and $\rho_c(z)$ and $\rho_h(z)$ are the densities in the cold and hot columns (kg/m³).

**Prediction.** For $H = 2500$ m, $\Delta P_b \approx$ ______ MPa.

In [ ]:
def column(T_of_P, H=2500.0, dz=5.0, P_top=P_ATM):
    '''Hydrostatic column stepped down from the water table.
    T_of_P: function returning temperature (C) for pressure (MPa).
    Returns depth z (m) and pressure P (MPa).'''
    z = np.arange(0.0, H + dz, dz)
    P = np.zeros_like(z); P[0] = P_top
    for i in range(len(z) - 1):
        T = T_of_P(P[i])
        if P[i] < P_CRIT and abs(T - saturation(P[i])["T"]) < 1e-6:
            rho = saturation(P[i])["rho_l"]            # on the boiling curve: saturated liquid
        else:
            rho = water(P[i], T)["rho"]
        P[i + 1] = ___
    return z, P


def T_cold(P):
    return 10.0


def T_hot(P, T_res=250.0):
    if P >= P_CRIT:
        return T_res
    return ___                                         # boiling-capped temperature


z_c, P_c = column(T_cold)
z_h, P_h = column(T_hot)
dP_b = P_c[-1] - P_h[-1]
print(f"Pressure at 2500 m: cold {P_c[-1]:.2f} MPa, hot {P_h[-1]:.2f} MPa, difference {dP_b:.2f} MPa")

plt.plot(P_c, z_c, label="cold 10 C"); plt.plot(P_h, z_h, label="hot 250 C")
plt.gca().invert_yaxis(); plt.xlabel("pressure (MPa)"); plt.ylabel("depth (m)"); plt.legend()
plt.show()

In [ ]:
# Test
assert 4.4 < dP_b < 4.9
print("B4.1 OK")

### B4.2 How much does it flow?

Assume half of $\Delta P_b$ drives fluid up through the upflow zone of height $H$:

$$q_{up} \approx \frac{k}{\mu}\,\frac{\Delta P_b}{2H}, \qquad \dot{m} = \rho\,q_{up}\,A_u, \qquad \dot{E} = \dot{m}\,(h_{up} - h_{rech})$$

where $A_u$ is the horizontal area of the upflow zone (m²), $\dot{m}$ is the upflow mass rate (kg/s), $\dot{E}$ is the heat carried by the upflow (W), $h_{up}$ is the specific enthalpy of the upflowing fluid and $h_{rech}$ that of the recharge (J/kg). Use fluid properties at 250 °C and 15 MPa, recharge at 10 °C, and $A_u = 1$ km².

In [ ]:
up = water(15, 250)
h_rech = water(0.1, 10)["h"]                 # kJ/kg
A_u = 1e6                                    # m2
H = 2500.0                                   # m
grad_excess = ___                            # Pa/m, half of dP_b over H

rows = []
for k in [1e-15, 1e-14, 1e-13]:              # m2
    q_up = ___                               # m/s
    m_dot = ___                              # kg/s
    E_MW = ___                               # MW
    rows.append({"k (m2)": f"{k:.0e}", "q_up (m/yr)": q_up * 3.156e7, "m_dot (kg/s)": m_dot, "heat (MW)": E_MW})
pd.DataFrame(rows).round(2)

In [ ]:
# Test
assert 60 < rows[1]["m_dot (kg/s)"] < 80
print("B4.2 OK")

### C4. Extensions

1. **Shared base pressure.** Start the hot column at the base with the cold-column base pressure and step **upward**. Plot $P_{hot} - P_{cold}$ against depth. At 500 m depth, which way does the horizontal pressure gradient push fluid? What does a pressure above atmospheric at the top of the hot column mean physically?
2. **Topography.** A recharge area lies on a plateau 300 m above the upflow area. Why must you compare pressures at the same elevation, not the same depth below ground?
3. **Darcy's own data.** Darcy (1856) measured flow $Q$ (L/min) through a sand column of length 0.58 m and diameter 0.35 m for head differences $\Delta h$ (m): Q = [3.60, 7.65, 12.0, 14.28, 15.2, 21.8, 23.41, 24.5, 27.8, 29.4], Δh = [1.11, 2.36, 4.00, 4.90, 5.02, 7.63, 8.13, 8.58, 9.86, 10.89]. Fit the hydraulic conductivity $K$ (m/s) through the origin and convert to permeability with $k = K\mu/(\rho g)$.

## Before you leave. Reflection

Answer in one or two sentences each.

1. Which equation did you solve in Part 3? Write it and define every symbol with its unit.
2. Name one assumption in Part 3 and one in Part 4.
3. One limiting case you checked.
4. One way these simple models could mislead a decision-maker.

---
# Assignment 1 (take-home)

**Hand in two files.**

1. **A written report (PDF), at most 8 pages including figures.** Typeset (LaTeX, Word equation editor, or Jupyter markdown exported to PDF) or neatly handwritten and scanned for derivations.
2. **This notebook (.ipynb)**, which must reproduce every number and figure in the report.

**Requirements for the report**

- Number every equation. **Define every symbol with its unit immediately after the equation where it first appears.**
- State the assumptions behind each model before using it.
- Give every numerical result with units and a sensible number of significant figures.
- Every figure has labelled axes with units and a caption of one or two sentences.
- For **at least one** problem, also include: the prediction you made before computing, a unit check, a limiting case, a judgement of whether the result is physically reasonable, one modification of the model, and one way the model could mislead a decision-maker.
- You may be asked to explain any code cell in a short oral check.

**Weights.** Problem 1, 25 %. Problem 2, 30 %. Problem 3, 25 %. Problem 4, 20 %.

## Problem 1. Enthalpy, heat capacity and latent heat (written)

**(a)** Starting from the first law for a closed system, $dU = \delta Q - P\,dV$, show that the heat added at constant pressure equals the change in $H = U + PV$. Define every symbol with its unit. Explain in two sentences why this makes enthalpy the natural variable for geothermal fluids.

**(b)** Compute the energy needed to take 1 kg of water at 1 MPa from 10 °C to saturated steam, in two steps (heating the liquid, then boiling it), and add a third step of superheating the steam to 300 °C. Report each step in kJ/kg and as a percentage of the total. How much of the latent heat is volume work $P\,\Delta v$?

**(c)** Repeat the liquid heating step in (b) three ways: with $c_p = 4.2$ kJ/kg/K, with $\int c_p\,dT$ from IF97, and from the tabulated enthalpy difference. Explain the differences. Over what temperature range is a constant $c_p$ good to better than 2 %?

**(d)** A well produces 45 kg/s with a flowing enthalpy of 1200 kJ/kg. The separated water is reinjected at 160 °C. Compute the thermal power extracted, relative to the reinjection enthalpy, and state your assumptions.

**(e)** Optional, for those who want the full flowing-fluid balance. It reads $\dot{Q} - \dot{W}_s = \dot{m}[\Delta h + \Delta(v^2/2) + g\,\Delta z]$. Estimate $g\,\Delta z$ for a 1500 m well and $v^2/2$ for a wellhead velocity of 40 m/s, both in kJ/kg, and compare with the latent heat at 1.5 MPa. What does this justify in Problem 2?

## Problem 2. Steam fraction in a producing well (notebook and written)

A well has a single feed zone at 1500 m depth. Liquid water enters the well at 290 °C. The flowing pressure is 10.0 MPa at the feed zone and 1.5 MPa at the wellhead. **Assume** the pressure varies linearly with depth between these two values.

**(a)** Calculate the depth at which boiling starts in the well.

**(b)** Calculate and plot the steam mass fraction $x$ and steam volume fraction $S$ against depth for three models: (i) isenthalpic; (ii) including potential energy; (iii) including potential energy and a heat loss of 150 W/m, for mass flow rates of 20 kg/s and 5 kg/s. Report the wellhead values of $x$ in a table.

**(c)** The measured wellhead enthalpy of this well is 1450 kJ/kg. Is that consistent with liquid at 290 °C entering the well? Discuss at least two physical processes, in the reservoir or in the well, that could explain the difference.

**(d)** Discuss whether a linear pressure profile is reasonable. How would the lower density of the two-phase mixture, and friction, change the shape of $P(z)$ above the boiling depth?

In [ ]:
# Problem 2 workspace (starter).  Given values:
z_fz = 1500.0          # m, feed zone depth
T_fz = 290.0           # C, feed temperature
P_fz = 10.0            # MPa, flowing pressure at the feed zone
P_wh = 1.5             # MPa, wellhead pressure
q_loss = 150.0         # W/m, heat loss per metre of well

def P_well(z):
    '''Assumed linear pressure profile in the well (MPa) at depth z (m).'''
    return P_wh + (P_fz - P_wh) * z / z_fz

# Your code here

## Problem 3. Clausius-Clapeyron and the BPD curve (notebook and written)

**(a)** Write the discrete BPD scheme (explicit Euler), define every symbol, and list its physical assumptions (at least four). Then show that combining the hydrostatic equation with Clausius-Clapeyron gives $dT/dz = \rho_l\, g\, T (v_v - v_l)/L$, and use it to explain why the BPD curve is steep near the surface and flat at depth. Support the explanation with computed values of $dT/dz$ at 100, 500, 1000 and 2000 m.

**(b)** Compute the BPD pressure at 1000 m depth with $\Delta z$ = 200, 100, 50, 10 and 1 m. Using $\Delta z$ = 1 m as reference, plot the absolute error against $\Delta z$ on log-log axes and find the slope. Explain what the slope tells you about the method.

**(c)** A clay cap at 400 m depth separates a shallow hydrostatic zone from a deeper confined reservoir whose pressure is 1.0 MPa higher than the BPD column just above the cap. Continue the BPD curve below 400 m from this higher pressure. What is the maximum liquid temperature at 1000 m? What overpressure at 400 m would allow liquid at 320 °C at 1000 m?

**(d)** Using the ideal-gas form of Clausius-Clapeyron with $L$ at 175 °C and a reference point at 100 °C, compute the boiling pressure at 150, 200, 250 and 300 °C and compare with IF97. Report the errors and explain where and why the approximation breaks down.

**(e)** Explain qualitatively how dissolved NaCl and dissolved CO₂ would each shift the BPD curve.

In [ ]:
# Problem 3 workspace

## Problem 4. Why does the reservoir flow? (written and notebook)

**(a)** Starting from Darcy's law in pressure form, derive the upward and horizontal flux components with depth $z$ positive downward. Define every symbol. Show that a hydrostatic column has no vertical flow, and state the condition for upflow.

**(b)** Compute $\Delta P_b$ for $H$ = 2500 m between a cold column at 10 °C and hot columns with $T_{res}$ = 250 °C and 300 °C, each capped by the boiling curve.

**(c)** Using the order-of-magnitude model from Part 4, compute $q_{up}$, $\dot{m}$ and $\dot{E}$ for $k$ = $10^{-15}$, $10^{-14}$ and $10^{-13}$ m² for both reservoir temperatures, with $A_u$ = 1 km². Which permeability gives a natural heat output of order 100 MW?

**(d)** List at least three assumptions in (c) and say for each whether relaxing it would increase or decrease the estimated heat output.

**(e)** If the hot and cold columns share the same pressure at 2500 m depth, what is the pressure difference between them at 500 m depth? If they are 2 km apart and connected by rock with $k = 10^{-14}$ m², estimate the horizontal Darcy flux and its direction. What feature at the surface would you expect above the hot column?

In [ ]:
# Problem 4 workspace